In [1]:
import requests
import pandas as pd
import json, time
import os
from pathlib import Path
import boto3, json

if Path.cwd().name == "notebooks":
    os.chdir("..")

from src.config import load_config

CONFIG = load_config()

In [2]:
iam         =  boto3.client("iam", region_name="eu-west-3")
role_name   = "mlflow-ec2-role2"
profil_name = "mlflow-ec2-profil2"
policy_name = "s3-mlflow-artifacts2"
group_name  = "mlflow-ec2-sg"
key_name    = "nappecast-ec2-key"

### Security group

In [ ]:
ec2 = boto3.client("ec2", region_name="eu-west-3")

vpc_id = ec2.describe_vpcs(Filters=[{"Name": "is-default", "Values": ["true"]}])["Vpcs"][0]["VpcId"]
my_ip = requests.get("https://checkip.amazonaws.com").text.strip()

mlflow_sg = ec2.create_security_group(GroupName=group_name, Description="MLflow server", VpcId=vpc_id)
mlflow_sg_id = mlflow_sg["GroupId"]

ec2.authorize_security_group_ingress(
    GroupId=mlflow_sg_id,
    IpPermissions=[
        {"IpProtocol": "tcp", "FromPort": 22, "ToPort": 22, "IpRanges": [{"CidrIp": f"{my_ip}/32"}]},
        {"IpProtocol": "tcp", "FromPort": 5000, "ToPort": 5000, "IpRanges": [{"CidrIp": f"{my_ip}/32"}]},
    ],
)
print(mlflow_sg_id)

### Rôle et policy

In [68]:

trust_policy = {
    "Version": "2012-10-17",
    "Statement": [{"Effect": "Allow", "Principal": {"Service": "ec2.amazonaws.com"}, "Action": "sts:AssumeRole"}],
}
iam.create_role(RoleName=role_name, AssumeRolePolicyDocument=json.dumps(trust_policy))
iam.attach_role_policy(
    RoleName=role_name,
    PolicyArn="arn:aws:iam::aws:policy/AmazonSSMManagedInstanceCore",
)

# pour ajouter les artifacts a s3
s3_policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["s3:GetObject", "s3:PutObject", "s3:ListBucket"],
        "Resource": [
            "arn:aws:s3:::nappecast",
            "arn:aws:s3:::nappecast/mlflow-artifacts/*",
            "arn:aws:s3:::nappecast/app/src/mlflow/*",
        ],
    }],
}
iam.put_role_policy(RoleName=role_name, PolicyName=policy_name, PolicyDocument=json.dumps(s3_policy))

iam.create_instance_profile(InstanceProfileName=profil_name)
iam.add_role_to_instance_profile(InstanceProfileName=profil_name, RoleName=role_name)

time.sleep(10)

### création de l'instance

In [ ]:
ssm = boto3.client("ssm", region_name="eu-west-3")
ami_id = ssm.get_parameter(Name="/aws/service/ami-amazon-linux-latest/al2023-ami-kernel-default-x86_64")["Parameter"]["Value"]

user_data = """#!/bin/bash
yum update -y

fallocate -l 1G /swapfile
chmod 600 /swapfile
mkswap /swapfile
swapon /swapfile
echo '/swapfile swap swap defaults 0 0' >> /etc/fstab

yum install -y docker
systemctl enable docker
systemctl start docker

mkdir -p /usr/local/lib/docker/cli-plugins
curl -SL https://github.com/docker/compose/releases/latest/download/docker-compose-linux-x86_64 -o /usr/local/lib/docker/cli-plugins/docker-compose
chmod +x /usr/local/lib/docker/cli-plugins/docker-compose

BUILDX_URL=$(curl -s https://api.github.com/repos/docker/buildx/releases/latest | grep "browser_download_url.*linux-amd64\\"" | cut -d '"' -f 4 | head -n 1)
curl -SL "$BUILDX_URL" -o /usr/local/lib/docker/cli-plugins/docker-buildx
chmod +x /usr/local/lib/docker/cli-plugins/docker-buildx

yum install -y python3-pip
pip3 install boto3
"""

resp = ec2.run_instances(
    ImageId=ami_id,
    InstanceType="t3.small",
    KeyName=key_name,
    SecurityGroupIds=[mlflow_sg_id],
    IamInstanceProfile={"Name": profil_name},
    UserData=user_data,
    MinCount=1, MaxCount=1,
    TagSpecifications=[{"ResourceType": "instance", "Tags": [{"Key": "Name", "Value": "mlflow-server-v3"}]}],
)
mlflow_instance_id = resp["Instances"][0]["InstanceId"]
ec2.get_waiter("instance_running").wait(InstanceIds=[mlflow_instance_id])

mlflow_ip = ec2.describe_instances(InstanceIds=[mlflow_instance_id])["Reservations"][0]["Instances"][0]["PublicIpAddress"]
print("IP :", mlflow_ip, "| instance_id :", mlflow_instance_id)

IP : 13.37.105.252 | instance_id : i-06b315e78fa277e77


### Copie des fichiers vers s3

In [4]:
s3_session = boto3.Session()
s3 = s3_session.client("s3")

import os
from pathlib import Path

EXCLUDE_DIRS = {".git", "__pycache__", ".venv", "node_modules"}
EXCLUDE_FILES = {".env"}

project_root = Path("..").resolve() / "NappeCast"   # depuis notebook/, remonte à la racine du projet

def upload_path(s3_client, path: Path, bucket, s3_prefix, base: Path):
    if not path.exists():
        print(f"⚠️  Introuvable, ignoré : {path}")
        return
    if path.is_file():
        relative = path.relative_to(base).as_posix()
        print(f"Upload : {path} -> s3://{bucket}/{s3_prefix}/{relative}")
        s3_client.upload_file(str(path), bucket, f"{s3_prefix}/{relative}")
    elif path.is_dir():
        for root, dirs, files in os.walk(path):
            dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS]
            for file in files:
                if file in EXCLUDE_FILES:
                    continue
                local_file = Path(root) / file
                relative = local_file.relative_to(base).as_posix()
                print(f"Upload : {local_file} -> s3://{bucket}/{s3_prefix}/{relative}")
                s3_client.upload_file(str(local_file), bucket, f"{s3_prefix}/{relative}")

# upload to s3
upload_path(s3, project_root / "src" / "mlflow" / "Dockerfile", "nappecast", "app", base=project_root)
upload_path(s3, project_root / "src" / "mlflow" / "docker-compose.yml", "nappecast", "app", base=project_root)

Upload : /home/ronanguilloueee/NappeCast/src/mlflow/Dockerfile -> s3://nappecast/app/src/mlflow/Dockerfile
Upload : /home/ronanguilloueee/NappeCast/src/mlflow/docker-compose.yml -> s3://nappecast/app/src/mlflow/docker-compose.yml


### connexion EC2
ssh -i "nappecast-ec2-key.pem" ec2-user@ec2-51-45-56-7.eu-west-3.compute.amazonaws.com

bash

cat > /home/ec2-user/download_s3.py << 'EOF'
import boto3, os

s3 = boto3.client("s3", region_name="eu-west-3")
bucket = "nappecast"
prefix = "app/src/mlflow/"
local_dir = "/home/ec2-user/mlflow"

paginator = s3.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
    for obj in page.get("Contents", []):
        key = obj["Key"]
        relative_path = key[len(prefix):]
        if not relative_path:
            continue
        local_path = os.path.join(local_dir, relative_path)
        os.makedirs(os.path.dirname(local_path), exist_ok=True)
        s3.download_file(bucket, key, local_path)
        print(f"Téléchargé : {key} -> {local_path}")
EOF

pip3 install boto3
python3 /home/ec2-user/download_s3.py


### build de l'instance docker
python3 /home/ec2-user/download_s3.py  
cd /home/ec2-user/mlflow  
sudo docker compose up -d --build  

### acces aux evènements du container
sudo docker compose logs mlflow --tail=50

In [1]:
resp = ec2.describe_instances(
    Filters=[{"Name": "tag:Name", "Values": ["mlflow-server-v2"]}, {"Name": "instance-state-name", "Values": ["running"]}]
)
mlflow_instance_id = resp["Reservations"][0]["Instances"][0]["InstanceId"]
mlflow_ip = resp["Reservations"][0]["Instances"][0]["PublicIpAddress"]
print(mlflow_instance_id, mlflow_ip)

NameError: name 'ec2' is not defined

### Ajout de nouvelles IP

In [ ]:
resp = ec2.describe_instances(
    Filters=[{"Name": "tag:Name", "Values": ["mlflow-server-v2"]}, {"Name": "instance-state-name", "Values": ["running"]}]
)
mlflow_sg_id = resp["Reservations"][0]["Instances"][0]["SecurityGroups"][0]["GroupId"]
print(mlflow_sg_id)

ec2.authorize_security_group_ingress(
    GroupId=mlflow_sg_id,
    IpPermissions=[
        {"IpProtocol": "tcp", "FromPort": 5000, "ToPort": 5000, "IpRanges": [{"CidrIp": "83.114.10.235/32"}]},
        {"IpProtocol": "tcp", "FromPort": 5000, "ToPort": 5000, "IpRanges": [{"CidrIp": "89.2.16.76/32"}]},
    ],
)

### Ajout de l'IP elastic

In [8]:
alloc = ec2.allocate_address(Domain="vpc")
ec2.associate_address(
    InstanceId=mlflow_instance_id,
    AllocationId=alloc["AllocationId"],
)
print("IP Elastique MLflow :", alloc["PublicIp"])

IP Elastique MLflow : 51.45.56.7


In [4]:
%pip install mlflow

  Using cached mlflow-3.14.0-py3-none-any.whl.metadata (49 kB)
  Using cached mlflow_skinny-3.14.0-py3-none-any.whl.metadata (50 kB)
  Using cached mlflow_tracing-3.14.0-py3-none-any.whl.metadata (19 kB)
  Using cached flask_cors-6.0.5-py3-none-any.whl.metadata (5.4 kB)
  Using cached flask-3.1.3-py3-none-any.whl.metadata (3.2 kB)
  Using cached alembic-1.18.5-py3-none-any.whl.metadata (7.2 kB)
  Using cached cryptography-48.0.1-cp311-abi3-manylinux_2_34_x86_64.whl.metadata (4.3 kB)
  Using cached graphene-3.4.3-py2.py3-none-any.whl.metadata (6.9 kB)
  Using cached gunicorn-26.0.0-py3-none-any.whl.metadata (5.4 kB)
  Using cached pandas-2.3.3-cp313-cp313-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached pyarrow-24.0.0-cp313-cp313-manylinux_2_28_x86_64.whl.metadata (3.0 kB)
  Using cached scikit_learn-1.9.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (11 kB)
  Using cached skops-0.14.0-py3-none-any.whl.metadata (4.4 kB)
  Using cach

In [1]:
import mlflow

mlflow.set_tracking_uri("http://51.45.56.7:5000")  # externe, depuis ton poste
client = mlflow.tracking.MlflowClient()

for rm in client.search_registered_models():
    print("Nom :", rm.name)
    for alias, version in (rm.aliases or {}).items():
        print("   alias:", alias, "-> version", version)

In [2]:
runs = client.search_runs(experiment_ids=[e.experiment_id for e in client.search_experiments()])
for r in runs:
    print(r.info.run_id, r.info.experiment_id, r.data.tags.get("mlflow.runName"))

bfebbfe67b124880958a4a37f7fa766c 1 prophet-forecast-best-H30
ad685f78ff6c4d78b519905c69548494 1 prophet-forecast-H30-cfg15
bd8c7834f09d49aeaf636cd2acc8e924 1 prophet-forecast-H30-cfg14
e8274c26dc42445cbf2a8871f38d08d6 1 prophet-forecast-H30-cfg13
d938dee4311e4f628f4e1783cb445655 1 prophet-forecast-H30-cfg12
41586359f7024eddb405c79ec37fa615 1 prophet-forecast-H30-cfg11
36eab0d83b47417289ecd43ba58c10ca 1 prophet-forecast-H30-cfg10
25a712ed0ad74b3baeee3f0b82402a79 1 prophet-forecast-H30-cfg9
e1c16ecc49954aefaca71cd905c86108 1 prophet-forecast-H30-cfg8
db6dfe0281294a3dac46f15ebeb4e148 1 prophet-forecast-H30-cfg7
edf431bc6daf4edbaf222f03e679ee69 1 prophet-forecast-H30-cfg6
28efa6eebeab4290a771ffc5d0f4fa79 1 prophet-forecast-H30-cfg5
232d66953cb742adb8f855a1faa51084 1 prophet-forecast-H30-cfg4
b1c973bfabc54dc18871762089120279 1 prophet-forecast-H30-cfg3
5d8025b52622427c8ce0d29f63801d82 1 prophet-forecast-H30-cfg2
107ceedab8eb4c3faffa32452f9d8068 1 prophet-forecast-H30-cfg1
51bc8cbc966a4726b5

In [3]:
run_id='bfebbfe67b124880958a4a37f7fa766c'

mlflow.register_model(f"runs:/{run_id}/model", "nappecast_Prophet")
client.set_registered_model_alias("nappecast_Prophet", "production", version=1)

Successfully registered model 'nappecast_Prophet'.


MlflowException: Unable to find a logged_model with artifact_path model under run bfebbfe67b124880958a4a37f7fa766c

In [5]:
artifacts = client.list_artifacts(run_id)
for a in artifacts:
    print(a.path, "| dossier:", a.is_dir)

forecast_best_H30.png | dossier: False
prediction_best_H30.png | dossier: False


In [6]:
client.set_registered_model_alias("nappecast_Prophet", "production", version=1)